# 🏭 Catálogo THC → Base de Datos

Extrae productos del catálogo PDF de [THC Chile](https://thc.cl) usando **OCR con GPU**.

## ¿Qué hace este notebook?
1. 📥 Descarga el catálogo PDF
2. 🧠 OCR página por página con PaddleOCR (GPU Tesla T4)
3. 🔍 Detecta productos: código SKU, nombre, precio
4. 📁 Exporta a CSV + Excel (descargables)
5. 🔎 Buscador interactivo de productos
6. 🖼️ Extrae imágenes embebidas (para fase de clasificación)

---
**Instrucciones:** Ejecuta las celdas en orden con `Shift+Enter`

In [ ]:
# ⚙️ Dependencias
!pip install pymupdf pandas openpyxl ipywidgets tqdm -q
!pip install paddlepaddle-gpu paddleocr -q

In [ ]:
import fitz
import pandas as pd
import requests
from PIL import Image, ImageDraw
import io, os, re, json, warnings
from tqdm.notebook import tqdm
from IPython.display import display, Image as IPImage, HTML
import ipywidgets as widgets

warnings.filterwarnings('ignore')
print("✅ Imports listos")

---
## 📥 1. Descargar catálogo

In [ ]:
URL = "https://thc.cl/archivos/catalogo-thc-chile-2026.pdf"
PDF_PATH = "catalogo-thc-chile-2026.pdf"

if not os.path.exists(PDF_PATH):
    print("Descargando catálogo...")
    r = requests.get(URL, stream=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(PDF_PATH, 'wb') as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192), total=total//8192, unit='KB'):
            f.write(chunk)
    print(f"Descargado: {os.path.getsize(PDF_PATH)/1024/1024:.1f} MB")
else:
    print(f"Ya existe: {os.path.getsize(PDF_PATH)/1024/1024:.1f} MB")

---
## 🔍 2. Inspeccionar PDF

In [ ]:
pdf = fitz.open(PDF_PATH)
print(f"📄 Páginas: {pdf.page_count}")
print(f"📏 Formato: {pdf[0].rect.width:.0f} x {pdf[0].rect.height:.0f} pts")

# Mostrar portada
page = pdf[0]
pix = page.get_pixmap(dpi=100)
display(IPImage(pix.tobytes("png"), width=600))

In [ ]:
# Verificar si el PDF tiene texto nativo o es imagen
sample_text = pdf[0].get_text()
if sample_text.strip():
    print("✅ El PDF tiene texto nativo (no necesita OCR)")
    print(f"   Ejemplo: {sample_text[:200]}...")
else:
    print("❌ Sin texto nativo → OCR necesario (bueno, tenemos GPU)")

---
## 🧠 3. Inicializar PaddleOCR (GPU)

In [ ]:
from paddleocr import PaddleOCR

print("⏳ Inicializando PaddleOCR...")
try:
    ocr = PaddleOCR(use_angle_cls=True, lang='es', use_gpu=True, show_log=False)
    print("✅ PaddleOCR listo en GPU")
except Exception as e:
    print(f"⚠️ GPU no disponible, usando CPU: {e}")
    ocr = PaddleOCR(use_angle_cls=True, lang='es', use_gpu=False, show_log=False)
    print("✅ PaddleOCR listo en CPU (será más lento)")

---
## 🔎 4. Ejecutar OCR página por página

Renderiza cada página a imagen y la procesa con PaddleOCR.
Esto toma **2-5 minutos** para ~50-80 páginas.

In [ ]:
os.makedirs("ocr_pages", exist_ok=True)
all_text = {}

for i in tqdm(range(pdf.page_count), desc="OCR"):
    page = pdf[i]
    pix = page.get_pixmap(dpi=200)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='PNG')
    result = ocr.ocr(img_bytes.getvalue())
    all_text[i] = result
    
    # Backup por si se cae la sesión
    with open(f"ocr_pages/page_{i:04d}.json", "w") as f:
        json.dump(result, f, ensure_ascii=False)

total_textos = sum(len(r[0]) for r in all_text.values() if r and r[0])
print(f"✅ OCR completado: {total_textos} textos detectados en {pdf.page_count} páginas")

In [ ]:
### ⏩ Si ya ejecutaste el OCR antes, recarga desde los JSON:
# for i in range(pdf.page_count):
#     with open(f"ocr_pages/page_{i:04d}.json") as f:
#         all_text[i] = json.load(f)

---
## 👁️ 5. Explorar resultados OCR

Mueve el slider para ver qué texto detectó el OCR en cada página.

In [ ]:
def show_page(page_num):
    result = all_text.get(page_num)
    if not result or not result[0]:
        display(HTML("<p>⚠️ Sin texto detectado</p>"))
        return
    
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=150)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    draw = ImageDraw.Draw(img)
    
    for line in result[0]:
        bbox = line[0]
        text = line[1][0]
        conf = line[1][1]
        draw.rectangle(bbox, outline="red", width=2)
        draw.text((bbox[0], bbox[1]-12), f"{conf:.0%}", fill="red")
    
    display(img)
    print(f"{'TEXTO':45s} {'CONF':>8s}")
    print("-" * 55)
    for line in sorted(result[0], key=lambda x: (x[0][1], x[0][0])):
        t = line[1][0][:50]
        print(f"{t:45s} {line[1][1]:6.0%}")

widgets.interact(show_page, page_num=widgets.IntSlider(0, 0, pdf.page_count-1, step=1))

---
## 📐 6. Analizar estructura del catálogo

Agrupa los textos detectados por filas (misma coordenada Y) para entender cómo están organizados los productos en la página.

In [ ]:
ROW_TOLERANCE = 25  # píxeles de tolerancia para agrupar en misma fila

def get_rows(page_num):
    result = all_text.get(page_num)
    if not result or not result[0]:
        return []
    
    items = []
    for line in result[0]:
        bbox = line[0]
        text = line[1][0].strip()
        if text:
            items.append({
                'x': bbox[0], 'y': bbox[1],
                'x2': bbox[2], 'y2': bbox[3],
                'text': text, 'conf': line[1][1]
            })
    
    items.sort(key=lambda it: (it['y'], it['x']))
    
    rows = []
    if not items:
        return rows
    current_row = [items[0]]
    for it in items[1:]:
        if abs(it['y'] - current_row[-1]['y']) < ROW_TOLERANCE:
            current_row.append(it)
        else:
            rows.append(current_row)
            current_row = [it]
    if current_row:
        rows.append(current_row)
    return rows

def show_structure(page_num):
    rows = get_rows(page_num)
    print(f"📄 Página {page_num+1}: {len(rows)} filas detectadas\n")
    for idx, row in enumerate(rows):
        texts = [f"{it['text']}(x={it['x']:.0f})" for it in sorted(row, key=lambda r: r['x'])]
        print(f"  Fila {idx+1:2d} (y≈{row[0]['y']:.0f}): {' | '.join(texts)}")

widgets.interact(show_structure, page_num=widgets.IntSlider(0, 0, pdf.page_count-1, step=1))

---
## 🔧 7. Parsear productos

Usa patrones para detectar:
- **Código SKU**: letras + números (ej: `THC-123`, `P045`)
- **Precio**: formato chileno (ej: `$12.990`)
- **Nombre**: el texto restante

Si ves muchos productos mal detectados, ajusta los parámetros abajo.

In [ ]:
# Parámetros ajustables
ROW_TOLERANCE = 25
SKU_PATTERN = re.compile(r'[A-Z]{1,4}[-]?\d{3,6}', re.IGNORECASE)
PRICE_PATTERN = re.compile(r'\$[\d\.]+')

def parse_products(all_text, page_count):
    products = []
    
    for page_num in range(page_count):
        result = all_text.get(page_num)
        if not result or not result[0]:
            continue
        
        # Build items with coordinates
        items = []
        for line in result[0]:
            bbox = line[0]
            text = line[1][0].strip()
            if not text:
                continue
            items.append({
                'x': bbox[0], 'y': bbox[1],
                'x2': bbox[2], 'y2': bbox[3],
                'text': text, 'conf': line[1][1],
                'page': page_num + 1
            })
        
        items.sort(key=lambda it: (it['y'], it['x']))
        
        # Group into rows by Y proximity
        rows = []
        if not items:
            continue
        current_row = [items[0]]
        for it in items[1:]:
            if abs(it['y'] - current_row[-1]['y']) < ROW_TOLERANCE:
                current_row.append(it)
            else:
                rows.append(current_row)
                current_row = [it]
        if current_row:
            rows.append(current_row)
        
        # Detect product fields in each row
        for row in rows:
            row.sort(key=lambda it: it['x'])
            sku = None
            price = None
            name_parts = []
            
            for it in row:
                t = it['text']
                m_sku = SKU_PATTERN.search(t)
                m_price = PRICE_PATTERN.search(t)
                
                if m_sku and not sku:
                    sku = m_sku.group()
                    remaining = t.replace(m_sku.group(), '', 1).strip()
                    if remaining:
                        name_parts.append(remaining)
                elif m_price and not price:
                    price = m_price.group()
                else:
                    name_parts.append(t)
            
            name = ' '.join(name_parts).strip()
            # Clean up repeated spaces
            name = re.sub(r'\s+', ' ', name)
            
            if sku or (price and name):
                products.append({
                    'pagina': page_num + 1,
                    'codigo': sku or '',
                    'nombre': name,
                    'precio': price or '',
                    'confianza': round(min(it['conf'] for it in row), 2)
                })
    
    return products

# Ejecutar parser
products = parse_products(all_text, pdf.page_count)
df = pd.DataFrame(products)
print(f"✅ {len(df)} productos detectados")
print(f"\nPrimeros 10:")
display(df.head(10))

# Estadísticas
if len(df) > 0:
    print(f"\n📊 Estadísticas:")
    print(f"   Con código: {df['codigo'].str.len().gt(0).sum()}")
    print(f"   Con precio: {df['precio'].str.len().gt(0).sum()}")
    print(f"   Confianza promedio: {df['confianza'].mean():.0%}")

---
## 📁 8. Exportar a CSV + Excel

In [ ]:
if len(df) > 0:
    # Exportar CSV
    df.to_csv("catalogo_thc_productos.csv", index=False, encoding='utf-8-sig')
    print("✅ catalogo_thc_productos.csv")
    
    # Exportar Excel
    df.to_excel("catalogo_thc_productos.xlsx", index=False, engine='openpyxl')
    print("✅ catalogo_thc_productos.xlsx")
    
    # Descargar al navegador
    from google.colab import files
    files.download("catalogo_thc_productos.csv")
else:
    print("⚠️ No hay productos para exportar. Ajusta los parámetros de parseo.")

---
## 🔍 9. Buscador interactivo

In [ ]:
if len(df) > 0:
    @widgets.interact
    def search(termino='', codigo='', mostrar=widgets.Dropdown(options=[10, 20, 50, 100], value=20, description='Mostrar')):
        mask = pd.Series([True] * len(df))
        if termino:
            mask &= df['nombre'].str.contains(termino, case=False, na=False)
        if codigo:
            mask &= df['codigo'].str.contains(codigo, case=False, na=False)
        
        result = df[mask]
        print(f"{len(result)} productos encontrados")
        display(result.head(mostrar))
else:
    print("⚠️ No hay datos. Ejecuta la celda de parseo primero.")

---
## 🖼️ 10. Extraer imágenes embebidas

Extrae las fotos de productos incrustadas en el PDF.
Esto prepara el terreno para clasificarlas con un modelo de IA (fase 2).

In [ ]:
IMG_DIR = "imagenes_extraidas"
os.makedirs(IMG_DIR, exist_ok=True)

total_images = 0
for page_num in tqdm(range(pdf.page_count), desc="Extrayendo imágenes"):
    page = pdf[page_num]
    image_list = page.get_images(full=True)
    
    page_dir = os.path.join(IMG_DIR, f"pagina_{page_num+1:04d}")
    os.makedirs(page_dir, exist_ok=True)
    
    for idx, img_info in enumerate(image_list):
        xref = img_info[0]
        base_image = pdf.extract_image(xref)
        img_bytes = base_image["image"]
        ext = base_image["ext"]
        
        fname = f"img_{idx:03d}.{ext}"
        with open(os.path.join(page_dir, fname), "wb") as f:
            f.write(img_bytes)
        total_images += 1

print(f"✅ {total_images} imágenes extraídas en {IMG_DIR}/")

if total_images > 0:
    print("\n📦 Comprimiendo para descargar...")
    !zip -r imagenes_extraidas.zip {IMG_DIR}/ -q
    files.download("imagenes_extraidas.zip")
    print("✅ Descarga iniciada")

---
## 🎯 Resumen

| Archivo | Contenido |
|---------|----------|
| `catalogo_thc_productos.csv` | Productos detectados (CSV UTF-8) |
| `catalogo_thc_productos.xlsx` | Productos detectados (Excel) |
| `imagenes_extraidas/` | Imágenes por página |
| `ocr_pages/` | OCR crudo por página (backup) |

---
### 🔜 Fase 2: Clasificar imágenes con IA

Cuando estés listo, usamos las imágenes extraídas + un modelo pre-entrenado
(ResNet18) para inferir la categoría del producto (taladro, llave, martillo, etc.)
y enriquecer la base de datos.